# 🛰️ SpaceShield AI — Interactive Analysis Notebook

This notebook demonstrates the full SpaceShield AI pipeline interactively.

**Sections:**
1. Setup & Imports
2. Generate Space Object Catalogue
3. Orbit Visualization
4. Conjunction Screening
5. ML Threat Classification
6. Risk Scoring
7. Maneuver Recommendations
8. Summary Statistics & Plots

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120

print('✅ Imports OK')

## 1. Generate Space Object Catalogue

In [ ]:
from src.debris_generator import DebrisGenerator

gen = DebrisGenerator(seed=42)
objects = gen.generate(n_active=40, n_debris=100, n_rockets=15, n_defunct=20)

df_cat = gen.to_dataframe(objects)
print(f'Total objects: {len(df_cat)}')
df_cat.head(10)

In [ ]:
# Altitude & inclination statistics
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Altitude
df_cat.groupby('object_type')['altitude_km'].hist(
    ax=axes[0], bins=30, alpha=0.7, legend=True)
axes[0].set_xlabel('Altitude [km]')
axes[0].set_title('Altitude Distribution by Object Type')

# Inclination
df_cat['inclination_deg'].hist(ax=axes[1], bins=36, color='steelblue', alpha=0.8)
axes[1].set_xlabel('Inclination [°]')
axes[1].set_title('Inclination Distribution')

# Object types
df_cat['object_type'].value_counts().plot.pie(ax=axes[2], autopct='%1.0f%%')
axes[2].set_title('Object Type Breakdown')
axes[2].set_ylabel('')

plt.tight_layout()
plt.show()

## 2. Conjunction Screening & Collision Probability

In [ ]:
from src.collision_prediction import ConjunctionScreener

screener = ConjunctionScreener(
    screening_threshold_km=15.0,
    conjunction_threshold_km=7.0
)

events = screener.screen(objects, duration_hours=12.0, verbose=True)
df_events = screener.to_dataframe(events)
print(f'\nConjunction events detected: {len(events)}')
if not df_events.empty:
    df_events.head()

In [ ]:
if events:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Pc distribution
    pcs = [ev.pc for ev in events if ev.pc > 0]
    if pcs:
        axes[0].hist(np.log10(pcs), bins=20, color='tomato', alpha=0.85)
        axes[0].set_xlabel('log₁₀(Collision Probability)')
        axes[0].set_ylabel('Count')
        axes[0].set_title('Collision Probability Distribution')
        axes[0].axvline(-5, color='green', ls='--', label='Green (1e-5)')
        axes[0].axvline(-4, color='orange', ls='--', label='Yellow (1e-4)')
        axes[0].legend(fontsize=8)

    # Miss distance vs lead time
    miss = [ev.miss_distance_km for ev in events]
    lead = [ev.lead_time_hours for ev in events]
    col  = ['red' if ev.risk_level in ('RED','ORANGE') else 'steelblue' for ev in events]
    axes[1].scatter(lead, miss, c=col, alpha=0.7, s=40)
    axes[1].set_xlabel('Lead Time [h]')
    axes[1].set_ylabel('Miss Distance [km]')
    axes[1].set_title('Miss Distance vs Lead Time')
    axes[1].axhline(1.0, color='red', ls='--', alpha=0.5, label='1 km')
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print('No events detected — try increasing screening threshold.')

## 3. ML Threat Classification

In [ ]:
from src.threat_classifier import ThreatClassifier

clf = ThreatClassifier(model_type='random_forest')
metrics = clf.train(n_samples=6000, verbose=True)

In [ ]:
# Feature importances
fi = clf.feature_importance()
if fi is not None:
    fi.sort_values().plot.barh(figsize=(8, 5), color='steelblue')
    plt.title('Feature Importances — Random Forest Threat Classifier')
    plt.xlabel('Importance (Mean Decrease in Impurity)')
    plt.tight_layout()
    plt.show()

if events:
    ml_labels = clf.predict(events)
    from collections import Counter
    print('\nML Threat Distribution:', dict(Counter(ml_labels)))

## 4. Composite Risk Scoring

In [ ]:
from src.risk_engine import RiskEngine

active_ids = {o.object_id for o in objects if o.active}
engine = RiskEngine()
scores = engine.score_events(events, active_ids=active_ids,
                             threat_labels=ml_labels if events else None)

df_scores = engine.to_dataframe(scores)
stats = engine.summary_stats(scores)
print('Risk Statistics:')
for k, v in stats.items():
    print(f'  {k}: {v}')

if not df_scores.empty:
    df_scores[['event_id','primary_id','secondary_id',
               'risk_score','risk_band','risk_grade','threat_level']].head(10)

## 5. Maneuver Recommendations

In [ ]:
from src.maneuver_recommender import ManeuverRecommender

obj_masses = {o.object_id: o.mass_kg for o in objects}
recommender = ManeuverRecommender(target_pc=1e-5)
recs = recommender.recommend(events, scores, object_masses=obj_masses)

recommender.print_summary(recs)

df_recs = recommender.to_dataframe(recs)
if not df_recs.empty:
    df_recs[['event_id','primary_id','maneuver_type',
             'delta_v_ms','fuel_cost_kg','new_pc','urgent']].head(10)

In [ ]:
# Delta-v distribution
active_recs = [r for r in recs if r.delta_v_ms > 0]
if active_recs:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    dvs = [r.delta_v_ms for r in active_recs]
    axes[0].hist(dvs, bins=15, color='darkorange', alpha=0.85)
    axes[0].set_xlabel('Δv [m/s]')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Maneuver Δv Distribution')

    from collections import Counter
    type_counts = Counter(r.maneuver_type for r in active_recs)
    axes[1].pie(type_counts.values(), labels=type_counts.keys(), autopct='%1.0f%%')
    axes[1].set_title('Maneuver Type Distribution')

    plt.tight_layout()
    plt.show()
else:
    print('No active maneuvers required.')

## 6. Generate Full Report

In [ ]:
import os
os.makedirs('../results', exist_ok=True)
os.makedirs('../data', exist_ok=True)

gen.save_catalogue(objects, '../data/catalogue.csv')
screener.save_events(events, '../results/close_approaches.csv')
engine.save(scores, '../results/risk_scores.csv')
recommender.save(recs, '../results/maneuver_recommendations.csv')

print('✅ All data files saved to results/ and data/')

In [ ]:
# Final summary table
print('=' * 55)
print('  SPACESHIELD AI — NOTEBOOK SUMMARY')
print('=' * 55)
print(f'  Objects tracked     : {len(objects)}')
print(f'  Conjunctions found  : {len(events)}')
print(f'  ML accuracy         : {metrics["accuracy"]:.4f}')
print(f'  ML F1 (weighted)    : {metrics["f1_weighted"]:.4f}')
print(f'  Risk scores         : {len(scores)}')
if stats:
    print(f'  Mean risk score     : {stats["mean_score"]:.1f}/100')
    print(f'  Max risk score      : {stats["max_score"]:.1f}/100')
print(f'  Maneuver recs       : {len(recs)}')
print(f'  Urgent maneuvers    : {sum(1 for r in recs if r.urgent)}')
print('=' * 55)